# Neural Scaling Laws and Compute-Optimal Training

Plot a language model's test loss against its parameter count $N$, its training tokens $D$, or its training compute
$C$, on log–log axes, and the points fall near straight lines that bend toward a floor. Straight on log–log means a
power law, $L \approx E + A/N^{\alpha}$. Two questions follow. Where do the exponents come from? And, given a budget of
compute, how should we split it between a bigger model and more data?

We answer the first with two models where the exponents can be proved: a memorizer on Zipf-distributed text (the
corpus's own Pitman–Yor generator) and a linear model with a power-law spectrum trained by SGD. We answer the second
with Lagrange's method on the fitted law, and we check both on a grid of toy transformers trained on the finance
corpus. The exponents are theorems only inside the models; for real language models they are fits, and the fits
disagree with one another more than their error bars admit.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "scaling_laws.py").exists() else here / "notebooks" / "scaling-laws"))
import scaling_laws as sl

S = sl.summary()
print("solvable model: fitted alpha, beta =", round(S["lin_alpha"], 3), round(S["lin_beta"], 3),
      "| predicted", sl.LIN_B - 1, (sl.LIN_B - 1) / sl.LIN_A)
print("N* ~ C^a: Approaches 1, 2, 3 =", round(S["lin_a1"], 3), round(S["lin_a2"], 3), round(S["lin_a3"], 3),
      "| predicted", 1 / (1 + sl.LIN_A))
print("toy transformers: a (Approach 2) =", round(S["toy_a_chinchilla"], 3), "| Kaplan's count:",
      round(S["toy_a_kaplan"], 3), "| Approach 3:", round(S["toy_a3"], 3))

## 1. Data scaling from Zipf: the missing mass

A *memorizer* predicts a token's type only if it has seen that type before. Its error after $n$ tokens is the
probability that token $n + 1$ has a new type: the **missing mass**. The finance corpus draws its content words from
Pitman–Yor restaurants $\mathrm{PY}(d, \theta)$ with $d = 0.6$ and $\theta = 20$: a new type arrives with probability
$(\theta + d K_n)/(\theta + n)$, where $K_n$ is the number of types so far.

**Thm 1.** For $\mathrm{PY}(d, \theta)$ with $0 \le d < 1$ and $\theta > 0$,
$$\Pr[\text{draw } n+1 \text{ is new}] = \frac{\theta + d\,\mathbb E K_n}{\theta + n}
= \frac{\Gamma(\theta+1)\,\Gamma(\theta+d+n)}{\Gamma(\theta+d)\,\Gamma(\theta+n+1)} \sim
\frac{\Gamma(\theta+1)}{\Gamma(\theta+d)}\, n^{-(1-d)}.$$

*Proof.* The rule is linear in $K_n$, so taking expectations gives the first equality. The same rule gives
$\mathbb E K_{n+1} + \theta/d = (1 + d/(\theta+n))(\mathbb E K_n + \theta/d)$, a product that telescopes into Gamma
functions (Topic 1's closed form), and Gamma-function ratios give the asymptote. ∎

The ranked frequencies of a PY measure decay as $i^{-1/d}$, so Zipf's exponent is $s = 1/d$. Hutter's lookup learner
on Zipf($s$) data errs at rate $n^{-(1 - 1/s)} = n^{-(1-d)}$, the same exponent. At $d = 0$ (a Dirichlet process)
the rate is $\theta/(\theta+n)$ and $\beta = 1$, the case Hutter's §6 describes.

In [ ]:
sl.test_missing_mass_exact()
sl.test_dirichlet_case()
sl.test_missing_mass_exponent()
sim = sl.py_simulation()
for n, f, se, ex in zip(sim["n"], sim["new"], sim["new_se"], sl.py_missing_mass(sim["n"])):
    print(f"n = {n:5d}   simulated {f:.4f} ± {se:.4f}   exact {ex:.4f}")

The corpus's text mixes seven restaurants with closed-class template words that saturate early. Past ten
thousand tokens its new-type rate falls with slope near $-(1-d) = -0.4$.

In [ ]:
sl.test_corpus_learning_curve()
c = sl.corpus_learning_curve()
print("tokens", c["tokens"], "| types", c["types"], "| fitted slope past 10k tokens:", round(c["slope"], 4))

## 2. Model-size scaling: the tail of a spectrum

The solvable model: features $x \sim \mathcal N(0, \Lambda)$ in $\mathbb R^M$ with $\lambda_k = k^{-a}$, a target
$y = \langle w^*, x\rangle$ with $\lambda_k (w^*_k)^2 = k^{-b}$, and a size-$N$ model that fits the first $N$
coordinates. Here $a = 1.5$, $b = 2.5$, $M = 8192$. Coordinate $k$ contributes $\lambda_k (w_k - w^*_k)^2$ to the loss, so
even a perfectly trained size-$N$ model keeps the **tail** $\sum_{k > N} k^{-b}$.

**Thm 2.** $\displaystyle \int_{N+1}^{M+1} x^{-b}\,dx \;\le\; \sum_{k=N+1}^{M} k^{-b} \;\le\; \int_{N}^{M} x^{-b}\,dx$,
so as $M \to \infty$ the tail is $N^{-(b-1)}/(b-1)$ up to a factor $(1 + 1/N)^{-(b-1)}$.

*Proof.* $x^{-b}$ is decreasing, so $(k+1)^{-b} \le \int_k^{k+1} x^{-b}\,dx \le k^{-b}$; sum over $k$. ∎

In [ ]:
sl.test_tail_bracket()
for N in (16, 64, 256):
    lo, hi = sl.tail_bracket(N, sl.LIN_B, sl.LIN_M)
    print(f"N = {N:4d}   {lo:.6e} <= tail {sl.tail_mass(N):.6e} <= {hi:.6e}")

## 3. Data scaling: gradient flow and exact SGD

Gradient flow from $w = 0$ on the population loss moves each coordinate as $w_k(t) - w^*_k = -w^*_k e^{-\lambda_k t}$.
The loss at time $t$ is $\sum_k k^{-b} e^{-2 t k^{-a}}$: the small-$k$ (large-eigenvalue) directions are learned
first, and the ones with $k \gtrsim t^{1/a}$ are still untouched.

**Thm 3.** With $s = (b-1)/a$, $\displaystyle\sum_{k \ge 1} k^{-b} e^{-2t k^{-a}} = \frac{\Gamma(s)}{a}(2t)^{-s}\,(1 + o(1))$
as $t \to \infty$.

*Proof.* Substitute $u = 2t x^{-a}$: $\int_0^\infty x^{-b} e^{-2t x^{-a}}\,dx = \frac{1}{a}(2t)^{-s}\Gamma(s)$. The
summand is unimodal in $x$, so the sum and the integral over $[1, \infty)$ differ by at most its maximum,
$(b/(2ta))^{b/a} e^{-b/a} = O(t^{-b/a})$, which is $o(t^{-s})$. The integral over $(0, 1)$ is $O(e^{-2t})$. ∎

One-pass SGD with step $\eta$ follows gradient flow at time $t = \eta D$, plus noise. For Gaussian features the noise
is computable exactly.

**Prop 1.** Let $e = w - w^*_{1:N}$ and $S_k = \mathbb E[e_k^2]$. One SGD step on a fresh sample gives
$S_k \leftarrow (1 - 2\eta\lambda_k + 2\eta^2\lambda_k^2) S_k + \eta^2 \lambda_k\big(\sum_j \lambda_j S_j + \text{tail}(N)\big)$.

*Proof.* Isserlis: $\mathbb E[x x^\top A x x^\top] = 2\Lambda A \Lambda + \operatorname{tr}(\Lambda A)\Lambda$ for symmetric
$A$. The coordinates past $N$ are independent of the kept ones, so they act as noise of variance tail$(N)$. Only diagonal
entries enter the right-hand side. ∎

In [ ]:
sl.test_gradient_flow_asymptote()
sl.test_sgd_recursion_matches_monte_carlo()
for t in (10.0, 100.0, 1000.0):
    print(f"t = {t:6.0f}   residual / asymptote = {sl.gf_residual(t, M=10**6) / sl.gf_asymptote(t):.6f}")

## 4. Counting compute

A matmul of an $m \times k$ by a $k \times n$ matrix costs $2mkn$ FLOPs. Per token, each weight matrix is multiplied
once in the forward pass and twice in the backward pass (the input gradient and the weight gradient).

**Prop 2.** One training step of the toy decoder ($L$ pre-LN blocks of width $d$, context $n$, a tied $V \times d$
embedding) costs exactly $B n\,(6N + 12 L n d)$ matmul FLOPs on a batch of $B$ windows, where $N = 12 L d^2 + V d$.

*Proof.* Each block's weights are $12 d^2$: four $d \times d$ attention maps and the MLP's $d \times 4d$ and $4d \times
d$. The tied unembedding is one more $d \times V$ product. That gives $2N$ forward and $4N$ backward. Attention's
$QK^\top$ and $AV$ cost $2nd$ each per token per layer forward and twice that backward: $12 L n d$. ∎

We count by instrumenting the arrays themselves, so every matmul the code actually takes is counted.

In [ ]:
sl.test_decoder_gradients()
sl.test_flop_count()
for d in sl.TOY_WIDTHS:
    print(f"d = {d:2d}   N = {sl.matmul_params(d):6d}   counted = predicted = {sl.counted_step_flops(d):.4e}"
          f"   attention / 6N = {sl.attention_share(d):.3f}")

## 5. Compute-optimal allocation

**Thm 4.** Let $L(N, D) = E + A N^{-\alpha} + B D^{-\beta}$, with $A, B, \alpha, \beta > 0$. On the budget $6ND = C$,
the unique minimizer is
$$N^* = G\,(C/6)^{\beta/(\alpha+\beta)}, \qquad D^* = G^{-1}(C/6)^{\alpha/(\alpha+\beta)}, \qquad
G = \Big(\frac{\alpha A}{\beta B}\Big)^{1/(\alpha+\beta)},$$
and the frontier is $L^*(C) = E + K\,C^{-\alpha\beta/(\alpha+\beta)}$. The budget's Lagrange multiplier,
$\nu = -dL^*/dC$, is the loss bought by one more FLOP.

*Proof.* Put $D = C/(6N)$ and $u = \log N$: $f(u) = E + A e^{-\alpha u} + B (C/6)^{-\beta} e^{\beta u}$ is strictly
convex, and $f'(u) = 0$ gives $\alpha A N^{-\alpha} = \beta B D^{-\beta}$: at the optimum, the two terms trade at
the ratio of their exponents. Solve. ∎

For the solvable model the theorems give $\alpha = b - 1$ and $\beta = (b-1)/a$, so $N^* \propto C^{1/(1+a)} = C^{0.4}$,
whatever $b$ is. (Here one SGD step costs $\propto N$, so $C = ND$.)

In [ ]:
sl.test_compute_optimal_closed_form()
sl.test_linear_scaling_exponents()
f = sl.lin_fits()
print("Approach 3 fit:", f["fit"])
print("a: Approach 1 =", round(f["a1"], 4), "| Approach 2 =", round(f["a2"], 4), "| Approach 3 =",
      round(f["a3"], 4), "| Thm 4 predicts", f["a_pred"])

The fitted law recovers the exponents to within 3%, and yet it is not the model's loss surface. A size-$N$ model's
data term sums only its $N$ kept coordinates, so at small $N$ the additive form overstates the loss.

In [ ]:
sl.test_additive_form_is_approximate()
print("log-residual of Eq. 2: rms", round(f["resid_rms"], 4), "| max", round(f["resid_max"], 4))

**Published fits.** Hoffmann et al. print $E = 1.69$, $A = 406.4$, $B = 410.7$, $\alpha = 0.34$, $\beta = 0.28$.
Besiroglu et al. recovered the unrounded values from the paper's source, and refit the data reconstructed from its
figures. The rounding alone moves the optimal size at Gopher's budget from 40B to 32B parameters.

In [ ]:
sl.test_published_fits()
for k, fit in sl.PUBLISHED_FITS.items():
    o = sl.compute_optimal(fit, sl.GOPHER_C)
    print(f"{k:20s} a = {fit.a:.4f}   N* = {o['N'] / 1e9:5.1f}B   D* = {o['D'] / 1e12:.2f}T   tokens/param = "
          f"{o['tokens_per_param']:.1f}")

## 6. Toy transformers on the finance corpus

Two pre-LN blocks of width $d \in \{6, \dots, 48\}$, learned positions, and tied embeddings over the 511 commonest
types plus [UNK]. Each model is trained once at each of three budgets $C = 6ND$, with the learning-rate schedule
ending at the run's own horizon. Each budget's losses against $\log N$ form a valley, and the parabola through them
puts its vertex at $N^*$ (Approach 2).

In [ ]:
sl.test_seed_spread()
sl.test_toy_isoflop_profiles()
for r in sl.isoflop_grid():
    print(f"C = {r['budget']:.2g}   d = {r['d']:2d}   N = {r['N']:6d}   D = {r['D']:8d}   loss = {r['loss']:.4f}")
tf = sl.toy_fits()
print("N* per budget:", np.round(tf["N_star"]), "| a =", round(tf["a_chinchilla"], 4), "| Approach 3 a =",
      round(tf["a3"], 4), "| one point's seed sd:", round(sl.seed_spread()["sd"], 4))

## 7. Kaplan against Chinchilla

Kaplan et al. found $N^* \propto C^{0.73}$; Hoffmann et al. found $C^{0.50}$. Kaplan counted only non-embedding
parameters $N_{\text{ne}}$ and compute $6 N_{\text{ne}} D$.

**Prop 3.** If $N^* \propto C^a$ in total parameters and $\rho = d\log N_{\text{ne}}/d\log N$ along the model family, then
the non-embedding optimum has local exponent $d\log N^*_{\text{ne}}/d\log C_{\text{ne}} = \rho a/(\rho a + 1 - a)$.

*Proof.* Along the optimum, $d\log N^*_{\text{ne}} = \rho a\, d\log C$ and $d\log D^* = (1 - a)\,d\log C$, and
$C_{\text{ne}} = 6 N^*_{\text{ne}} D^*$. Divide. ∎

Embeddings grow as $N_{\text{ne}}^{1/3}$ in a family of fixed aspect ratio, so $\rho$ rises from 1 toward 3 as the
embedding's share grows. At $a = 0.5$ the exponent then rises toward $3a/(1+2a) = 0.75$. So the same physics, read in
Kaplan's units at small scale, gives Kaplan's number (Pearce & Song's reconciliation).

In [ ]:
sl.test_kaplan_counting_shift()
k = sl.kaplan_shift()
for C_, loc, pred, sh in list(zip(k["C"], k["local"], k["predicted"], k["share"]))[::4]:
    print(f"C = {C_:.1e}   embedding share {sh:.3f}   local exponent {loc:.4f}   Prop 3 {pred:.4f}")
print("regression below 1e18 FLOPs:", round(k["a_small"], 4), "| limit 3a/(1+2a) =", round(k["limit"], 4))

## 8. Finance case study: a finite filing corpus

A firm that trains on its own filings has a fixed number $U$ of unique tokens. Muennighoff et al. fit how much a
repeated token is worth: $D$ tokens drawn from $U$ unique ones count as
$D' = U + U R^*(1 - e^{-R/R^*})$ fresh tokens, with $R = D/U - 1$ repeated epochs and $R^* \approx 15.4$. We spend the
same compute on 1, 4, 16, and 64 epochs of a shrinking corpus.

In [ ]:
sl.test_repetition()
rep = sl.repetition_runs()
for r in rep["rows"]:
    print(f"epochs {r['epochs']:3d}   unique {r['unique']:7d}   loss {r['loss']:.4f}   D' = {r['D_eff']:9.0f}"
          f"   toy fit at D' {r['pred']:.4f}")

At the scale of a finance model, a data cap barely binds at first. With the compute of BloombergGPT's run
(50.6B parameters on 569B of its 709B tokens), Besiroglu's fit wants a little more data than exists, and the optimum
repeats 1.04 epochs, which Muennighoff's decay makes nearly free. Shrink the corpus and the cap starts to bind.

In [ ]:
sl.test_data_cap_moves_the_optimum()
cap = sl.cap_sweep()
for U, N, ep in list(zip(cap["U"], cap["N"], cap["epochs"]))[::4]:
    print(f"U = {U / 1e9:7.1f}B unique tokens   N* = {N / 1e9:5.1f}B   epochs = {ep:5.2f}")
print("unconstrained N* =", round(cap["free"]["N"] / 1e9, 2), "B on", round(cap["free"]["D"] / 1e9), "B tokens")

## The lab's data and the harness

In [ ]:
sl.test_viz_payload()
sl.test_viz_mirror_matches_payload()
sl.test_corpus_fingerprint()
print("payload sections:", list(sl.viz_payload()))

In [ ]:
assert sl._run_harness() == 0